# IMPORTAÇÕES
Faz a importação das bibliotecas necessárias

In [89]:
import mariadb
import toml
import sys
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
import requests
import traceback
import unicodedata
import pandas as pd

# CONECTAR DB

## db_opas
Conecta ao banco de dados "opas_vigilancia_sindromica".

Os dados da conexão são extraídos do arquivo config.toml

In [90]:
def conectar_db_opas():
    
    config = toml.load(r"config.toml")

    db_opas = config['db_opas']

    try:
        conn = mariadb.connect(
            user = db_opas['user'],
            password = db_opas['password'],
            host = db_opas['host'],
            port = db_opas['port'],
            database = db_opas['db_name']
    )

        cursor = conn.cursor()
        return conn, cursor
    
    except mariadb.Error as e:
        print(f"Erro ao conectar ao MariaDB: {e}")
        sys.exit(1)

# FUNÇÕES

## salvar_sintomas()
Vai salvar no banco de dados da OPAS os sintomas e os dados do atendimento/paciente

In [91]:
def salvar_sintomas(prontuario, sintomas):
    # Limpeza da resposta do LLM
    sintomas = sintomas.replace("```json", "").replace("```", "").strip()

    #Conecta ao banco de dados
    conn, cursor = conectar_db_opas()

    sql = """INSERT INTO sintomas 
                (nr_atendimento, unidade_atendimento, data_hora_atendimento, dt_nascimento, sg_sexo, 
                cid10, evolucao, bairro, municipio, uf, sintomas)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ON DUPLICATE KEY UPDATE 
                    unidade_atendimento = VALUES(unidade_atendimento),
                    data_hora_atendimento = VALUES(data_hora_atendimento),
                    dt_nascimento = VALUES(dt_nascimento),
                    sg_sexo = VALUES(sg_sexo),
                    cid10 = VALUES(cid10),
                    evolucao = VALUES(evolucao),
                    bairro = VALUES(bairro),
                    municipio = VALUES(municipio),
                    uf = VALUES(uf),
                    sintomas = VALUES(sintomas)"""

    valores = (
        prontuario['nr_atendimento'], 
        prontuario['unidade'],
        prontuario['data_hora_atendimento'], 
        prontuario['dt_nascimento'], 
        prontuario['sg_sexo'], 
        prontuario['cid10'], 
        prontuario['evolucao'], 
        prontuario['bairro'],
        prontuario['municipio'],
        prontuario['uf'],
        sintomas,)

    try:
        cursor.execute(sql, valores)
        conn.commit()
    
    except mariadb.Error as err:
        print("\n========== ERRO DETALHADO MARIADB ==========")
        print(f"Código do erro: {err.errno}")
        print(f"SQLSTATE: {err.sqlstate}")
        print(f"Mensagem completa: {err}")
        print("\nValores enviados:")
        print(valores)
        print("============================================\n")
        conn.rollback()

    except Exception as e:
        print("\n========== ERRO INESPERADO ==========")
        print(str(e))
        print(traceback.format_exc())
        print("=====================================\n")
        conn.rollback()

    finally:
        cursor.close()
        conn.close()

# nome_modelo
Obtém do config.toml o nome do modelo de LLM a ser utilizado.

In [92]:
def nome_modelo():
    config = toml.load(r"config.toml")
    nome_modelo = config['llm']['nome_modelo']
    return nome_modelo

# url_llm

Obtém config.toml o URL do servidor do LLM

In [93]:
def url_llm():
    config = toml.load(r"config.toml")
    url = config['llm']['url']
    return url

## remover_acentos()
Remove os acentos e caracteres especiais do texto da evolução

In [94]:
def remover_acentos(texto):

    texto_normalizado = unicodedata.normalize('NFD', texto)
    
    # Filtra o texto mantendo apenas caracteres que não sejam sinais de acentuação (Mn)
    texto_sem_acentos = "".join(
        char for char in texto_normalizado 
        if unicodedata.category(char) != 'Mn').upper()
    
    return texto_sem_acentos

## enviar_llm()
Recebe os prontuários e envia para a LLM extrair os sintomas.
Finaliza enviando os sintomas e o prontuário para salvar_sintomas(prontuario, sintomas) para que os dados sejam salvos no banco de dados.

In [ ]:
def enviar_llm(prontuarios):
  
  # Começa a iteração do "prontuarios"
  for i, prontuario in prontuarios.iterrows():

    # Calcula e exibe a % já executada
    porcentagem = round(((i+1)/prontuarios.shape[0])*100,2)
    print(f"\tLLM: executando {i+1}/{prontuarios.shape[0]} ({porcentagem}%) - {prontuario['nr_atendimento']}")
       

    # Promt para o LLM
    PROMPT = f""" 
        Você é um sistema de extração de dados clínicos para vigilância sindrômica. Sua prioridade máxima é a fidelidade aos dados PATOLÓGICOS e a exclusão rigorosa de achados de normalidade.

        ### TAREFA
        Extrair sinais e sintomas POSITIVOS e ALTERADOS do texto. Converta descrições clínicas para termos médicos canônicos.

        ### REGRAS DE OURO (OBRIGATÓRIAS)
        1. FILTRO DE NORMALIDADE (CRÍTICO): NÃO extraia achados que indiquem estado normal, estável ou ausência de sintoma. 
          - EXEMPLOS DE EXCLUSÃO PROIBIDA: "corado", "hidratado", "eupneico", "anictérico", "acianótico", "indolor", "sem déficit", "RHA+", "BNF 2T", "Glasgow 15", "MUV+", "SRA", "normotenso", "afebril", "estável", "sem placas".

        2. FILTRO DE NEGATIVAS: Ignore tudo o que o paciente "nega" ou que está descrito como "ausente".

        3. NOMENCLATURA CANÔNICA (OBRIGATÓRIA): Use o termo médico padrão. Corrija erros ortográficos ao converter para o termo canônico.
          - Ex: "Otorreia" (secreção no ouvido), "Otalgia" (dor de ouvido), "Odinofagia" (dor de garganta), "Cefaleia" (dor de cabeça), "Epixtaxe" (sangramento nasal), "Mialgia" (dor no corpo).

        4. ATOMIZAÇÃO ESTRITA: É PROIBIDO agrupar múltiplos sintomas em um único objeto. 
          - Exemplo: "Febre, tosse e dor" deve gerar TRÊS entradas.

        5. ESCOPO: Ignore diagnósticos (ex: "Dengue", "HAS"), medicações e condutas. Foque no quadro agudo atual.

        6. Sem textos introdutórios ou markdown

        7. A resposta deverá ser em formato de lista. Exemplo: ['sintoma 1', sintoma 2']

        8. Se não existir sintomas devolva nulo. Sem qualquer comentário.

        9. Resposta SOMENTE em português.

        10. Antes de finalizar certifique-se que os sintomas estão em formato canônico da medicina.

        TEXTO DE ENTRADA:
        {prontuario['evolucao']}"""

    # Define a URL do LLM
    url = rf"http://{url_llm()}:11434/api/generate"

    # Configurações para o LLM
    payload = {
        "model": nome_modelo(),
        "prompt": PROMPT,
        "stream": False,
        "options": {
        "num_ctx": 2048,}
    }

    inicio = time.time()

    try:
      # Envia para o LLM
      response = requests.post(url, json=payload, timeout=240)
      response.raise_for_status()

      # Recebe do LLM os sintomas do atendimento
      sintomas = response.json()["response"]
      print(f"\tTempo no llm: {time.time() - inicio:.2f}s\n")

      # Remove os acentos dos sintomas
      sintomas = remover_acentos(sintomas)

      #Salva os dados no banco de dados
      salvar_sintomas(prontuario, sintomas)
      
    except:
      print("\tNão foi possível obter a resposta do LLM")
      print("")

## obter_lista_nr_atendimento_db_opas()
Essa função obtém todos os códigos de prontuário que já foram processados.
Essa informação é utilizada para que o sistema não envie para o LLM duas vezes o mesmo prontuário.

In [96]:
def obter_lista_nr_atendimento_db_opas():
    conn, cursor = conectar_db_opas()

    sql = """SELECT nr_atendimento
            FROM sintomas"""
    
    cursor.execute(sql)
    prontuarios_ja_realizados = [linha[0] for linha in cursor.fetchall()]

    conn.close()

    return set(prontuarios_ja_realizados)

# tratar_prontuarios

Trata a variável "prontuarios" convertendo dicionário e lista de dicionário para formato DataFrame

In [ ]:
def tratar_prontuarios(prontuarios):
    # Verifica se já é um DataFrame
    if isinstance(prontuarios, pd.DataFrame):
        df = prontuarios.copy()
    
    # Se for um dicionário, converte diretamente
    elif isinstance(prontuarios, dict):
        df = pd.DataFrame([prontuarios])
    
    # Se for lista, verifica se é lista de dicionários
    elif isinstance(prontuarios, list):
        if all(isinstance(item, dict) for item in prontuarios):
            df = pd.DataFrame(prontuarios)
        else:
            raise ValueError("A lista deve conter apenas dicionários.")
    
    else:
        raise TypeError("prontuarios deve ser um DataFrame, dicionário ou lista de dicionários.")
    
    # Remove os já processados
    df = df[~df['nr_atendimento'].isin(obter_lista_nr_atendimento_db_opas())]

    return df

# MAIN

In [ ]:
def main():



    ######################################################
    #  INSERIR AQUI AS FUNÇÕES DE CONSULTA AO BANCO DE   # 
    #  DADOS QUE IRÁ FORNECER OS DADOS DO PRONTUÁRIO     #
    ######################################################



    # Mais de um prontuário pode ser enviado de uma vez.
    # Ele pode estar tanto em formato de lista de dicionário ou DataFrame(Pandas)

    # EXEMPLO de prontuario
    prontuarios = [
        {
        'nr_atendimento': '123456',  #Tem que estar em formato de Integer
        'unidade': 'UPA Sul',
        'data_hora_atendimento': '2025-12-29 01:30:00',
        'dt_nascimento': '2012-01-01',
        'sg_sexo': 'M',
        'cid10': 'J189',
        'evolucao': """<p>Paciente refere que estava com dor de garganta há 6 dias. Foi avaliada em unidade, prescrito siintomaticos. Evolui ha 2 dias com febre, tosse secretiva, rouquidão.</p>
                    <p>&nbsp.</p>
                    <p>Ao exame: REG, corada e hidratada</p>
                    <p>Oroscopia hiperemia</p>
                    <p>AR MV+ bilateralmente sem RA</p>
                    <p>&nbsp.</p>
                    <p>HD PAC atípica</p>
                    <p>CD orientações gerais</p>
                    <p>prescrevo azirtomicina por 5 dias e sintomaticos</p>
                    <p>orientações gerais, se sinais de alarme ou se não houver melhora clínica em até 72 horas do ATB</p>""",
        'uf': 'SC',
        'municipio': 'FLORIANOPOLIS',
        'bairro': 'RATONES',
    },
        {
        'nr_atendimento': '654321',
        'unidade': 'UPA Norte',
        'data_hora_atendimento': '2025-12-30 21:45:10',
        'dt_nascimento': '1985-01-01',
        'sg_sexo': 'F',
        'cid10': 'N390',
        'evolucao': """<p>Paciente com queixa de disúria há 3 dias, associada a polaciúria e urgência miccional.</p>
                    <p>Nega febre ou dor lombar.</p>
                    <p>&nbsp.</p>
                    <p>Ao exame: BEG, corada, hidratada</p>
                    <p>Abdome flácido, indolor à palpação</p>
                    <p>Sem sinais de Giordano</p>
                    <p>&nbsp.</p>
                    <p>HD: ITU baixa</p>
                    <p>CD: prescrito antibiótico e orientações gerais</p>
                    <p>Orientada a retornar em caso de piora ou ausência de melhora</p>""",
        'uf': 'SC',
        'municipio': 'FLORIANOPOLIS',
        'bairro': 'INGLESES',
        }]

    # Encaminha para o LLM
    enviar_llm(tratar_prontuarios(prontuarios))
    

In [99]:
if __name__ == '__main__':
    main()